# Arabic Written Dialect Scan with MARBERTv2
This notebook is configured for a full run by default using `IbrahimAmin/marbertv2-arabic-written-dialect-classifier`.

Main outputs:
- `predictions.jsonl`: one record per processed row
- `row_probabilities.jsonl`: per-row probabilities for later threshold sweeps
- `dialect_progress.log`: visible live progress log in the run folder
- `progress_latest.json`: visible latest machine-readable status in the run folder
- `.logs/dialect_progress.log`: legacy hidden compatibility copy
- `.logs/progress_latest.json`: legacy hidden compatibility copy
- `summary.json`: latest/final summary


In [ ]:
import json
from collections import deque
from pathlib import Path

import pandas as pd

In [ ]:
# Full-run config. This notebook defaults to non-smoke mode.

PYTHON_BIN = '/home/MohammadNabulsi/whisper/.venv/bin/python'
SCRIPT_PATH = Path('/home/MohammadNabulsi/whisper/dialect_identifiaction/arabic_text_dialect_scan_marbertv2_written.py')

DATA_ROOT = Path('/home/MohammadNabulsi/whisper/data')
ALLOWED_SOURCES = ['masc_c', 'qasr']
SOURCE_FILE_PREFIXES = {
    'masc_c': 'masc_c_only__',
    'qasr': 'processed_qasr_segments__',
}

SMOKE_MODE = False
SMOKE_MAX_SAMPLES = 32

OUTPUT_DIR = '/home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written_clean_masc_c_qasr'
MODEL_ID = 'IbrahimAmin/marbertv2-arabic-written-dialect-classifier'

UNKNOWN_THRESHOLD = 0.55
TARGET_LABEL = 'LEV'
TARGET_THRESHOLD = 0.40
MAX_LENGTH = 512
MAX_SAMPLES = SMOKE_MAX_SAMPLES if SMOKE_MODE else None
DEVICE = 'auto'
BATCH_SIZE = 256
LOG_EVERY = 100
RESUME = True

matching_files = sorted(
    path for path in DATA_ROOT.glob('*')
    if path.is_file()
    and path.suffix.lower() in {'.arrow', '.parquet'}
    and any(path.name.startswith(SOURCE_FILE_PREFIXES[source]) for source in ALLOWED_SOURCES)
)

print({
    'data_root': str(DATA_ROOT),
    'allowed_sources': ALLOWED_SOURCES,
    'smoke_mode': SMOKE_MODE,
    'max_samples': MAX_SAMPLES,
    'matching_file_count': len(matching_files),
    'sample_files': [p.name for p in matching_files[:5]],
})


In [ ]:
cmd = [
    PYTHON_BIN, str(SCRIPT_PATH),
    '--data-root', str(DATA_ROOT),
    '--output-dir', OUTPUT_DIR,
    '--model-id', MODEL_ID,
    '--device', DEVICE,
    '--unknown-threshold', str(UNKNOWN_THRESHOLD),
    '--target-label', TARGET_LABEL,
    '--target-threshold', str(TARGET_THRESHOLD),
    '--max-length', str(MAX_LENGTH),
    '--batch-size', str(BATCH_SIZE),
    '--log-every', str(LOG_EVERY),
]

for source in ALLOWED_SOURCES:
    cmd += ['--allowed-source', source]

if MAX_SAMPLES is not None:
    cmd += ['--max-samples', str(MAX_SAMPLES)]

if RESUME:
    cmd += ['--resume']

print(' '.join(cmd))


In [4]:
# Run the scan. Leave SMOKE_MODE=False for the full run, or switch it to True above for a quick validation run.
!{' '.join(cmd)}


Applied source filter: ['masc_c', 'qasr']
Found 721 shards.
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00000-of-00009.parquet__5ab0b5b7dc__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00001-of-00009.parquet__2b38af7d8d__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00002-of-00009.parquet__efb5ed1f66__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00003-of-00009.parquet__133ba07b54__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00004-of-00009.parquet__1defcdef6f__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00005-of-00009.parquet__37f2aee764__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00006-of-00009.parquet__b921a58b89__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only__data__test-00007-of-00009.parquet__3e9237e0b5__clean.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c

In [5]:
latest_path = Path(OUTPUT_DIR) / 'progress_latest.json'
legacy_latest_path = Path(OUTPUT_DIR) / '.logs' / 'progress_latest.json'
chosen_path = latest_path if latest_path.exists() else legacy_latest_path
if chosen_path.exists():
    summary = json.loads(chosen_path.read_text(encoding='utf-8'))
    summary['status_path'] = str(chosen_path)
else:
    summary = {
        'status': 'missing_progress',
        'message': 'Run the scan cell successfully first. progress_latest.json has not been written yet.',
        'expected_path': str(latest_path),
        'legacy_expected_path': str(legacy_latest_path),
    }
summary

{'time': '2026-06-27 15:23:44',
 'attempted_in_this_run': 1439327,
 'completed_total': 1439327,
 'completed_before_run': 0,
 'total_rows': 1494127,
 'overall_progress_percentage': 96.3323,
 'processed_ok_total': 1439327,
 'skipped_total': 0,
 'errors_total': 0,
 'current_file': 'DONE',
 'current_uid': 'DONE',
 'elapsed_seconds': 2621.42,
 'rows_per_second': 549.06,
 'estimated_remaining_seconds': 99.81,
 'estimated_remaining_human': '1m39s',
 'dialect_counts': {'MAGHREB': 53616,
  'LEV': 184709,
  'MSA': 975698,
  'GLF': 63991,
  'EGY': 49425,
  'unseen': 111888},
 'dialect_percentages': {'MAGHREB': 3.7251,
  'LEV': 12.833,
  'MSA': 67.7885,
  'GLF': 4.4459,
  'EGY': 3.4339,
  'unseen': 7.7736},
 'unseen_count': 111888,
 'unseen_percentage': 7.7736,
 'status_path': '/home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written_clean_masc_c_qasr/progress_latest.json'}

In [6]:
if 'dialect_percentages' in summary:
    pct = summary['dialect_percentages']
    counts = summary['dialect_counts']
    pd.DataFrame([
        {'label': label, 'count': counts[label], 'percentage': pct[label]}
        for label in counts
    ]).sort_values('percentage', ascending=False)
else:
    pd.DataFrame()

In [7]:
prob_path = Path(OUTPUT_DIR) / 'row_probabilities.jsonl'

if prob_path.exists():
    tail = deque(maxlen=5)
    with prob_path.open('r', encoding='utf-8') as f:
        for line in f:
            tail.append(json.loads(line))
    preview = []
    for rec in tail:
        preview.append({
            'uid': rec['uid'],
            'top_label': rec['top_label'],
            'top_score': rec['top_score'],
            'final_label': rec['final_label'],
            'target_label_score': rec.get('target_label_score'),
            'text': rec['text'][:160],
        })
    pd.DataFrame(preview)
else:
    pd.DataFrame()

In [8]:
# Threshold experiment for Levantine candidates without rerunning inference.

ANALYSIS_LABEL = 'LEV'
ANALYSIS_THRESHOLD = 0.40
TOP_K = 20

prob_path = Path(OUTPUT_DIR) / 'row_probabilities.jsonl'
matches = []
total_scored = 0

if prob_path.exists():
    with prob_path.open('r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            score = rec.get('label_scores', {}).get(ANALYSIS_LABEL)
            if score is None:
                continue
            total_scored += 1
            if score >= ANALYSIS_THRESHOLD:
                matches.append({
                    'uid': rec['uid'],
                    'score': score,
                    'top_label': rec['top_label'],
                    'final_label': rec['final_label'],
                    'text': rec['text'],
                })

match_df = pd.DataFrame(matches).sort_values('score', ascending=False).head(TOP_K) if matches else pd.DataFrame()
print({
    'analysis_label': ANALYSIS_LABEL,
    'analysis_threshold': ANALYSIS_THRESHOLD,
    'rows_scored': total_scored,
    'rows_meeting_threshold': len(matches),
})
match_df

{'analysis_label': 'LEV', 'analysis_threshold': 0.4, 'rows_scored': 1439327, 'rows_meeting_threshold': 218694}


,uid,score,top_label,final_label,text
35839,masc_c_only__data__train-00212-of-00400.parque...,0.999452,LEV,LEV,وكنت راجعه على بيتي وكنت عم بسمع صوت ام ماسكه ...
41197,masc_c_only__data__train-00245-of-00400.parque...,0.999445,LEV,LEV,عشان هيك كانت الحياه صعبه كتير
3168,masc_c_only__data__train-00014-of-00400.parque...,0.999444,LEV,LEV,وكانوا الجيران عم يسقوا الزريعه وريحه التراب ط...
44541,masc_c_only__data__train-00265-of-00400.parque...,0.999443,LEV,LEV,وقبل ما اوصل على البيت لقيت انه عم بشم ريحه ال...
57503,masc_c_only__data__train-00348-of-00400.parque...,0.999443,LEV,LEV,بس ما مان بدو يعمل هالمحاوله مع ادم بشكل مباشر
75267,processed_qasr_segments__train__shard_000016.a...,0.999442,LEV,LEV,لما عرفت بيسان بانه الوالد والوالد استشهدوا
218324,processed_qasr_segments__train__shard_000307.a...,0.999442,LEV,LEV,احك لي لا لا دقيقة انا بدي اسالك
5584,masc_c_only__data__train-00029-of-00400.parque...,0.999441,LEV,LEV,وبسبب الخطيه اللي عملوها فهني رح يختيروا ويموت...
48726,masc_c_only__data__train-00293-of-00400.parque...,0.999441,LEV,LEV,عم بتعلم نطق ولغه ومصطلحات جديده ما كان عندو ا...
33348,masc_c_only__data__train-00197-of-00400.parque...,0.999440,LEV,LEV,وبكل مسا كان الله يجي على الجنه تا يمشي ويحكي ...


In [9]:
log_path = Path(OUTPUT_DIR) / 'dialect_progress.log'
legacy_log_path = Path(OUTPUT_DIR) / '.logs' / 'dialect_progress.log'
chosen_log_path = log_path if log_path.exists() else legacy_log_path
print('Showing log from:', chosen_log_path)
!tail -n 20 "{chosen_log_path}"

Showing log from: /home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written_clean_masc_c_qasr/dialect_progress.log
[2026-06-27 15:23:37] progress=96.03% completed=1434874/1494127 ok=1434874 skipped=0 errors=0 rate=548.83 rows/s eta=1m47s unseen=7.78% MAGHREB=3.73% LEV=12.83% MSA=67.78% GLF=4.45% EGY=3.44% unseen=7.78% file=/home/MohammadNabulsi/whisper/data/processed_qasr_segments__train__shard_000306.arrow__30545b7f4b__clean.parquet uid=processed_qasr_segments__train__shard_000306.arrow__30545b7f4b__clean.parquet:uid=qasr:C00A1D54-5F0C-46BC-A83A-9755DA294AD3:C00A1D54_5F0C_46BC_A83A_9755DA294AD3_utt_77_align
[2026-06-27 15:23:37] progress=96.04% completed=1434995/1494127 ok=1434995 skipped=0 errors=0 rate=548.85 rows/s eta=1m47s unseen=7.78% MAGHREB=3.73% LEV=12.83% MSA=67.78% GLF=4.45% EGY=3.44% unseen=7.78% file=/home/MohammadNabulsi/whisper/data/processed_qasr_segments__train__shard_000306.arrow__30545b7f4b__clean.parquet uid=processed_qasr_segments__train__shard_000306